## nb_ingest_live_feed

**Hourly fake-ingestion notebook** — triggered by `pl_fake_ingestion` at :05 past each hour.

Reads the equivalent 2022 hour from the historical parquet (same month/day/hour, year remapped to 2022),
then writes two outputs to the Lakehouse:

| Output | Path | Contents |
|---|---|---|
| Class B actuals | `Files/live_feed/class_b/YYYY/MM/DD/HH/flights.parquet` | All columns incl. ArrDelay, DepDelay, BTS causes |
| Class A schedule | `Files/live_feed/class_a_schedule/YYYY/MM/DD/HH/schedule.parquet` | CRS*/Distance/Airline only — no actuals |
| Rolling buffer | `Files/live_feed/rolling_buffer/YYYY/MM/DD/HH/flights.parquet` | Same as Class B; trimmed to last `BUFFER_HOURS` |

**Prerequisites:**
- This notebook must have a default Lakehouse attached.
- `Files/raw/historical_2022/Combined_Flights_2022.parquet` must exist in the Lakehouse.
- Install `pyarrow>=14.0` (included in Fabric Runtime 1.2+ by default).

In [ ]:
# ── Parameters (overridden by Data Factory pipeline at runtime) ──────────
# When running manually, edit these values directly.
SOURCE_YEAR = 2022         # Year to remap to (the historical file we use as source)
BUFFER_HOURS = 168         # Rolling buffer retention window (7 days)
BACKFILL_OFFSET_HOURS = 0  # Set >0 to backfill: 1 = process (now - 1h), etc.
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
import os
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ── Paths ─────────────────────────────────────────────────────────────────
# Fabric mounts the default Lakehouse at /lakehouse/default/.
# All read/write operations use standard filesystem paths — no ADLS SDK needed.
LAKEHOUSE_ROOT = Path("/lakehouse/default/Files")

SOURCE_PARQUET = LAKEHOUSE_ROOT / "raw" / "historical_2022" / "Combined_Flights_2022.parquet"
CLASS_B_ROOT   = LAKEHOUSE_ROOT / "live_feed" / "class_b"
CLASS_A_ROOT   = LAKEHOUSE_ROOT / "live_feed" / "class_a_schedule"
BUFFER_ROOT    = LAKEHOUSE_ROOT / "live_feed" / "rolling_buffer"

if not SOURCE_PARQUET.exists():
    raise FileNotFoundError(
        f"Source parquet not found: {SOURCE_PARQUET}\n"
        "Upload Combined_Flights_2022.parquet to Files/raw/historical_2022/ first."
    )

# ── Column definitions (must match src/data/loader.py CSV_DTYPES) ─────────
CLASS_A_COLS = [
    "FlightDate", "Airline", "Origin", "Dest",
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "Month", "DayOfWeek", "DayofMonth", "Cancelled", "Diverted",
]
CLASS_B_EXTRA = [
    "DepTime", "ArrTime", "DepDelay", "ArrDelay",
    "DepDel15", "ArrDel15", "WheelsOff", "WheelsOn",
    "TaxiOut", "TaxiIn", "AirTime", "ActualElapsedTime",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]
CLASS_B_COLS = CLASS_A_COLS + CLASS_B_EXTRA

print(f"Source parquet: {SOURCE_PARQUET} ({SOURCE_PARQUET.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# ── 1. Determine which hour to process ───────────────────────────────────
now_utc   = pd.Timestamp.utcnow().floor("h") - pd.Timedelta(hours=BACKFILL_OFFSET_HOURS)
equiv_dt  = now_utc.replace(year=SOURCE_YEAR)  # remap year, keep month/day/hour

print(f"Processing hour : {now_utc} (real-world)")
print(f"Equivalent 2022 : {equiv_dt}")
print(f"CRSDepTime range: [{equiv_dt.hour * 100}, {(equiv_dt.hour + 1) * 100})")

In [ ]:
# ── 2. Read Class B (actuals) for the current hour ────────────────────────
# PyArrow filter pushdown — only scans the matching row groups.
# Never loads the full ~7M row file.
hour_start = equiv_dt.hour * 100       # e.g., 14:00 → 1400
hour_end   = (equiv_dt.hour + 1) * 100 # e.g., 15:00 → 1500

# Keep only columns that exist in the file (older parquets may lack BTS columns)
parquet_schema = pq.read_schema(SOURCE_PARQUET)
available_cols = set(parquet_schema.names)
b_cols_to_read = [c for c in CLASS_B_COLS if c in available_cols]
a_cols_to_read = [c for c in CLASS_A_COLS if c in available_cols]

table_b = pq.read_table(
    str(SOURCE_PARQUET),
    columns=b_cols_to_read,
    filters=[
        ("Month",      "=",  equiv_dt.month),
        ("DayofMonth", "=",  equiv_dt.day),
        ("CRSDepTime", ">=", hour_start),
        ("CRSDepTime", "<",  hour_end),
    ],
)

print(f"Class B rows read: {len(table_b):,}")
if len(table_b) == 0:
    print("WARNING: 0 rows — check that the parquet covers this month/day/hour.")

In [ ]:
# ── 3. Write Class B to live_feed landing zone ────────────────────────────
class_b_dir = CLASS_B_ROOT / f"{now_utc.year}/{now_utc.month:02d}/{now_utc.day:02d}/{now_utc.hour:02d}"
class_b_dir.mkdir(parents=True, exist_ok=True)
class_b_path = class_b_dir / "flights.parquet"

pq.write_table(table_b, str(class_b_path), compression="snappy")
print(f"Class B written : {class_b_path} ({class_b_path.stat().st_size / 1e6:.2f} MB)")

# ── 4. Write to rolling buffer (same data, separate tree) ─────────────────
# The rolling buffer is what the inference notebook reads.
# Having a separate copy means we can trim it without touching class_b history.
buffer_dir  = BUFFER_ROOT / f"{now_utc.year}/{now_utc.month:02d}/{now_utc.day:02d}/{now_utc.hour:02d}"
buffer_dir.mkdir(parents=True, exist_ok=True)
buffer_path = buffer_dir / "flights.parquet"

pq.write_table(table_b, str(buffer_path), compression="snappy")
print(f"Buffer written  : {buffer_path}")

In [ ]:
# ── 5. Build Class A schedule for next 8 hours and write ──────────────────
# Class A = schedule-only columns (CRS*, Distance, Airline, etc.).
# These are used by the inference notebook as Block G (exogenous future features).
# We read 9 hours total (current + next 8) in one filtered scan.

sched_equiv_start = equiv_dt.hour * 100
sched_equiv_end   = min((equiv_dt.hour + 9) * 100, 2400)  # cap at 23:59

filters_a = [
    ("Month",      "=",  equiv_dt.month),
    ("DayofMonth", "=",  equiv_dt.day),
    ("CRSDepTime", ">=", sched_equiv_start),
    ("CRSDepTime", "<",  sched_equiv_end),
]

# Handle midnight overflow: if current hour + 8 crosses into the next day,
# we need a second filter block for the next calendar day.
if equiv_dt.hour + 9 >= 24:
    next_day = equiv_dt + pd.Timedelta(days=1)
    overflow_end = ((equiv_dt.hour + 9) % 24) * 100
    table_a_overflow = pq.read_table(
        str(SOURCE_PARQUET),
        columns=a_cols_to_read,
        filters=[
            ("Month",      "=",  next_day.month),
            ("DayofMonth", "=",  next_day.day),
            ("CRSDepTime", ">=", 0),
            ("CRSDepTime", "<",  overflow_end),
        ],
    )
else:
    table_a_overflow = None

table_a_main = pq.read_table(
    str(SOURCE_PARQUET),
    columns=a_cols_to_read,
    filters=filters_a,
)

if table_a_overflow is not None:
    import pyarrow as pa
    table_a = pa.concat_tables([table_a_main, table_a_overflow])
else:
    table_a = table_a_main

class_a_dir  = CLASS_A_ROOT / f"{now_utc.year}/{now_utc.month:02d}/{now_utc.day:02d}/{now_utc.hour:02d}"
class_a_dir.mkdir(parents=True, exist_ok=True)
class_a_path = class_a_dir / "schedule.parquet"

pq.write_table(table_a, str(class_a_path), compression="snappy")
print(f"Class A written : {class_a_path} ({len(table_a):,} rows, {class_a_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# ── 6. Trim rolling buffer to last BUFFER_HOURS partitions ────────────────
# Walk the buffer tree and delete hourly partitions older than the cutoff.
# The buffer tree structure is YYYY/MM/DD/HH/ so we can sort lexicographically.

cutoff = now_utc - pd.Timedelta(hours=BUFFER_HOURS)
deleted = 0

if BUFFER_ROOT.exists():
    for year_dir in sorted(BUFFER_ROOT.iterdir()):
        for month_dir in sorted(year_dir.iterdir()):
            for day_dir in sorted(month_dir.iterdir()):
                for hour_dir in sorted(day_dir.iterdir()):
                    try:
                        partition_ts = pd.Timestamp(
                            year=int(year_dir.name),
                            month=int(month_dir.name),
                            day=int(day_dir.name),
                            hour=int(hour_dir.name),
                            tz="UTC",
                        )
                        if partition_ts < cutoff:
                            for f in hour_dir.iterdir():
                                f.unlink()
                            hour_dir.rmdir()
                            deleted += 1
                    except (ValueError, OSError):
                        pass  # skip unexpected directory names

print(f"Buffer trimmed  : {deleted} partition(s) older than {cutoff} removed")
print(f"Ingestion complete for hour: {now_utc}")